# GRPO（Group Relative Policy Optimization）

> 本文件已有实现，此处仅在头部补充原理概述，原实现 cell 保持不变。DeepSeek-Math/R1 提出。

## 1. 动机
PPO 需要一个 critic 网络估计 value baseline，critic 难训且占显存。GRPO **省掉 critic**，改用"对同一 prompt 采样一组 G 个回答，用组内奖励的相对值做 baseline"。

## 2. 流程
1. 对每个 prompt $q$，从 $\pi_{\theta_{old}}$ 采样一组 $\{o_1,\dots,o_G\}$，各算奖励 $r_i$（规则/RM）。
2. 组内归一化得优势（无 critic）：
$$A_i=\frac{r_i-\text{mean}(r_{1:G})}{\text{std}(r_{1:G})}$$
3. 用 PPO 的 clip 目标在该组上更新，并加 KL 散度罚 $\beta\,D_{KL}(\pi_\theta\|\pi_{ref})$ 防偏离过远。

## 3. 考察点
- 为什么省 critic 仍 work（组内均值是合理 baseline）
- KL 罚的作用与 reference model 冻结
- 与 PPO/DPO 的关系：GRPO 是 online、组内相对；DPO 是 offline、 pairwise


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from transformers import LlamaConfig, LlamaForCausalLM

# 设置随机种子以保证可复现性
torch.manual_seed(42)

# ============================================================
# GRPO (Group Relative Policy Optimization) 核心思想：
# 对同一个问题生成 G 个不同响应，用组内相对奖励作为优势函数，
# 完全去掉了 Critic 模型，大幅降低训练成本。
# 来源论文: DeepSeekMath (Shao et al., 2024)
# ============================================================

# 超参数定义
G = 4           # 每个问题生成的响应数量（组大小）
VOCAB_SIZE = 12 # 词表大小（演示用）
PROMPT_LEN = 6  # 提示长度
OUTPUT_LEN = 4  # 每个响应的生成长度

# 定义示例数据 - 对同一个提示生成 G=4 个响应
# input_ids: [G, prompt_len] - 同一问题重复 G 次
prompt = torch.tensor([[3, 5, 2, 8, 1, 4]])
input_ids = prompt.repeat(G, 1)                          # [G, 6] - 批量复制

# output_ids: [G, output_len] - G 个不同的生成响应
output_ids = torch.randint(0, VOCAB_SIZE, (G, OUTPUT_LEN))  # [G, 4]

# 拼接完整序列
full_ids = torch.cat([input_ids, output_ids], dim=1)     # [G, 10]
full_mask = torch.ones_like(full_ids)                    # [G, 10]

# 只在生成部分计算损失 - 构建响应掩码
response_mask = torch.zeros_like(full_ids)
response_mask[:, PROMPT_LEN:] = 1                        # [G, 10] - 仅标记生成部分

print("== 数据形状 ==")
print("提示ID形状:     ", input_ids.shape,    "# [G, prompt_len]")
print("响应ID形状:     ", output_ids.shape,   "# [G, output_len]")
print("完整序列形状:   ", full_ids.shape,     "# [G, seq_len]")
print("响应掩码形状:   ", response_mask.shape,"# [G, seq_len], 仅生成部分为1")

In [ ]:
# 创建策略模型和参考模型
# GRPO 与 PPO 的关键区别: 不需要 Critic 模型！
policy_model = LlamaForCausalLM(config=LlamaConfig(
    vocab_size=VOCAB_SIZE, num_hidden_layers=1, hidden_size=32
))
reference_model = deepcopy(policy_model)  # 深度复制，参数完全相同

# 冻结参考模型参数（参考模型不更新）
for param in reference_model.parameters():
    param.requires_grad = False

# 测试前向传播
with torch.no_grad():
    policy_outputs = policy_model(full_ids)
    ref_outputs = reference_model(full_ids)

print("== 模型输出形状 ==")
print("策略模型 logits 形状:", policy_outputs.logits.shape)  # [G, seq_len, vocab_size]
print("参考模型 logits 形状:", ref_outputs.logits.shape)
print()
print("PPO 需要 4 个模型: Actor + Critic + Reward + Reference")
print("GRPO 只需 2 个模型: Policy + Reference  ✓")

In [ ]:
# ============================================================
# 工具函数：对数概率计算
# ============================================================

def logprobs_from_logits(logits, labels):
    """从 logits 计算给定标签序列的对数概率"""
    print(f"对数概率计算 - 输入logits形状: {logits.shape}, 标签形状: {labels.shape}")
    
    # [G, seq_len, vocab] -> log softmax
    logp = F.log_softmax(logits, dim=-1)
    print(f"  对数概率分布形状: {logp.shape}")
    
    # 取出每个位置对应 token 的对数概率
    logp_labels = torch.gather(logp, dim=-1, index=labels.unsqueeze(-1))
    print(f"  收集标签对数概率形状: {logp_labels.shape}")
    
    result = logp_labels.squeeze(-1)  # [G, seq_len]
    print(f"  最终对数概率形状: {result.shape}")
    return result


def masked_mean(values, mask, dim=None):
    """计算掩码区域的均值"""
    if dim is not None:
        result = (values * mask).sum(dim=dim) / mask.sum(dim=dim).clamp(min=1e-8)
    else:
        result = (values * mask).sum() / mask.sum().clamp(min=1e-8)
    return result


# 计算策略模型和参考模型的 token 级对数概率
policy_logits = policy_outputs.logits
ref_logits = ref_outputs.logits

print("== 策略模型对数概率 ==")
policy_logprobs = logprobs_from_logits(policy_logits, full_ids)

print()
print("== 参考模型对数概率 ==")
with torch.no_grad():
    ref_logprobs = logprobs_from_logits(ref_logits, full_ids)

print()
print("策略模型对数概率（第0个响应）:", policy_logprobs[0].detach())
print("参考模型对数概率（第0个响应）:", ref_logprobs[0])

In [ ]:
# ============================================================
# 奖励函数
# GRPO 原论文中奖励由规则函数给出（如数学题答案正确性）
# 这里用简单 MLP 模拟，返回每个响应的标量奖励
# ============================================================

class RewardModel(nn.Module):
    """简单奖励模型：对完整序列打一个标量分"""
    def __init__(self, vocab_size=12, hidden_size=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, masks=None):
        x = self.embedding(input_ids)                  # [G, seq_len, hidden]
        print(f"奖励模型 - 嵌入输出形状: {x.shape}")

        outputs, _ = self.lstm(x)                      # [G, seq_len, hidden]
        print(f"奖励模型 - LSTM输出形状: {outputs.shape}")

        # 取最后一个有效 token 的隐状态
        if masks is not None:
            last_indices = masks.sum(dim=1).long() - 1       # [G]
            batch_indices = torch.arange(outputs.size(0))
            last_hidden = outputs[batch_indices, last_indices]  # [G, hidden]
        else:
            last_hidden = outputs[:, -1]               # [G, hidden]

        print(f"奖励模型 - 最后隐藏状态形状: {last_hidden.shape}")

        reward = self.head(last_hidden).squeeze(-1)    # [G]
        print(f"奖励模型 - 输出奖励形状: {reward.shape}")
        return reward


reward_model = RewardModel()

with torch.no_grad():
    rewards = reward_model(full_ids, full_mask)        # [G]

print()
print(f"G={G} 个响应的奖励分数: {rewards}")
print(f"奖励形状: {rewards.shape}  # [G]")

In [ ]:
# ============================================================
# GRPO 核心：组内相对优势估计
#
# PPO 用 Critic 估计 V(s) 作为基线
# GRPO 用同一问题 G 个响应的平均奖励作为基线
#
# 优势 A_i = (r_i - mean(r)) / std(r)
#
# 每个 token 使用其所属响应的整体优势（序列级别）
# ============================================================

def compute_group_advantages(rewards):
    """
    计算组内相对优势
    
    参数:
        rewards: [G] - 组内 G 个响应的标量奖励
    返回:
        advantages: [G] - 标准化后的优势
    """
    print(f"组优势计算 - 奖励形状: {rewards.shape}")
    print(f"原始奖励: {rewards.detach().tolist()}")

    group_mean = rewards.mean()
    group_std  = rewards.std() + 1e-8  # 防止除零

    print(f"组内奖励均值: {group_mean.item():.6f}")
    print(f"组内奖励标准差: {group_std.item():.6f}")

    # 标准化：使每组优势零均值、单位方差
    advantages = (rewards - group_mean) / group_std   # [G]
    print(f"标准化优势: {advantages.detach().tolist()}")
    print(f"优势形状: {advantages.shape}")
    return advantages


advantages = compute_group_advantages(rewards)         # [G]

# 将序列级优势扩展为 token 级：每个 token 共享其所在响应的优势值
# [G] -> [G, seq_len]
token_advantages = advantages.unsqueeze(1).expand_as(full_ids).float()
# 仅在生成部分（response_mask）上应用优势
token_advantages = token_advantages * response_mask

print()
print("== token 级优势 ==")
print(f"token 优势形状: {token_advantages.shape}  # [G, seq_len]")
print(f"第0个响应的 token 优势: {token_advantages[0].tolist()}")
print("注意：提示部分优势为0，仅生成部分参与梯度计算")

In [ ]:
# ============================================================
# KL 散度惩罚
# 与 PPO 相同：防止策略更新过快偏离参考模型
# KL(π_θ || π_ref) = Σ π_θ * (log π_θ - log π_ref)
# 近似形式（per-token）: log π_θ(t) - log π_ref(t)
# ============================================================

def compute_kl_penalty(policy_logprobs, ref_logprobs, mask):
    """
    计算 token 级 KL 惩罚

    参数:
        policy_logprobs: [G, seq_len] - 策略模型对数概率
        ref_logprobs:    [G, seq_len] - 参考模型对数概率
        mask:            [G, seq_len] - 有效位置掩码
    返回:
        kl: [G, seq_len] - token 级 KL 散度
        mean_kl: 标量 - 掩码均值
    """
    print(f"KL计算 - 策略对数概率形状: {policy_logprobs.shape}")
    print(f"KL计算 - 参考模型对数概率形状: {ref_logprobs.shape}")

    # per-token KL 近似: log π_θ - log π_ref
    kl = policy_logprobs - ref_logprobs               # [G, seq_len]
    print(f"KL散度形状: {kl.shape}")

    # 只在有效 token 上计算均值
    mean_kl = masked_mean(kl, mask)
    print(f"掩码 KL 均值: {mean_kl.item():.6f}")

    return kl, mean_kl


kl, mean_kl = compute_kl_penalty(policy_logprobs, ref_logprobs.detach(), response_mask)

print()
print("KL 散度（第0个响应，生成部分）:", kl[0, PROMPT_LEN:].detach().tolist())

In [ ]:
# ============================================================
# GRPO 损失函数
#
# L_GRPO = -E[ A_i * log π_θ(o_i|q) ] + β * KL(π_θ || π_ref)
#
# 其中：
#   - A_i 是组内相对优势（标量，per-response）
#   - log π_θ(o_i|q) = Σ_t log π_θ(o_t|q, o_<t)  （生成 token 之和）
#   - β 是 KL 系数
#
# 等价于带重要性采样的策略梯度（类似 PPO 但无裁剪或使用裁剪变体）
# ============================================================

def grpo_loss(policy_logprobs, ref_logprobs, token_advantages, response_mask,
              old_logprobs=None, kl_coef=0.04, clip_eps=0.2, use_clip=True):
    """
    计算 GRPO 损失

    参数:
        policy_logprobs:  [G, seq_len] - 当前策略对数概率（需要梯度）
        ref_logprobs:     [G, seq_len] - 参考模型对数概率
        token_advantages: [G, seq_len] - token 级优势（生成部分非零）
        response_mask:    [G, seq_len] - 生成部分掩码
        old_logprobs:     [G, seq_len] - 采样时策略对数概率（用于重要性采样）
                          若为 None，则假设 on-policy（old == policy）
        kl_coef:  float - KL 惩罚系数 β
        clip_eps: float - PPO 风格裁剪系数（use_clip=True 时生效）
        use_clip: bool  - 是否使用 PPO 裁剪（DeepSeek-R1 变体使用裁剪）
    """
    print("== GRPO 损失计算 ==")
    print(f"  policy_logprobs 形状: {policy_logprobs.shape}")
    print(f"  token_advantages 形状: {token_advantages.shape}")
    print(f"  response_mask 形状: {response_mask.shape}")

    # -----------------------------------------------------------
    # 1. 重要性采样比率（off-policy 场景）
    # r_t(θ) = π_θ(o_t) / π_θ_old(o_t) = exp(log π_θ - log π_θ_old)
    # on-policy 时 ratio = 1
    # -----------------------------------------------------------
    if old_logprobs is None:
        # on-policy: ratio = 1，直接使用对数概率
        log_ratio = torch.zeros_like(policy_logprobs)
        ratio = torch.ones_like(policy_logprobs)
        print("  模式: on-policy (ratio=1)")
    else:
        log_ratio = policy_logprobs - old_logprobs.detach()
        ratio = torch.exp(log_ratio)                  # [G, seq_len]
        print(f"  模式: off-policy, 重要性采样比率均值: {(ratio * response_mask).sum() / response_mask.sum():.4f}")

    print(f"  ratio 形状: {ratio.shape}")

    # -----------------------------------------------------------
    # 2. 策略损失（目标：最大化 A * log π，即最小化 -A * log π）
    # -----------------------------------------------------------
    if use_clip:
        # PPO 裁剪变体（DeepSeek-R1-Zero 使用此方式）
        pg_loss1 = -token_advantages * ratio
        pg_loss2 = -token_advantages * torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)
        pg_loss_token = torch.max(pg_loss1, pg_loss2)  # [G, seq_len]
        print(f"  使用 PPO 裁剪, clip_eps={clip_eps}")
    else:
        # 无裁剪变体（原始 GRPO 论文）
        # 直接用对数概率 × 优势（等价于 ratio × advantage 当 on-policy 时）
        pg_loss_token = -token_advantages * policy_logprobs   # [G, seq_len]
        print("  无裁剪变体（原始 GRPO）")

    # 仅在生成部分计算策略损失
    pg_loss = masked_mean(pg_loss_token, response_mask)
    print(f"  策略损失 (pg_loss): {pg_loss.item():.6f}")

    # -----------------------------------------------------------
    # 3. KL 散度损失
    # -----------------------------------------------------------
    kl_token = policy_logprobs - ref_logprobs.detach()  # [G, seq_len]
    kl_loss  = masked_mean(kl_token, response_mask)
    print(f"  KL 损失 (kl_loss): {kl_loss.item():.6f}")

    # -----------------------------------------------------------
    # 4. 总损失
    # -----------------------------------------------------------
    total_loss = pg_loss + kl_coef * kl_loss
    print(f"  总损失 = pg_loss + {kl_coef} * kl_loss = {total_loss.item():.6f}")

    stats = {
        "pg_loss":    pg_loss.item(),
        "kl_loss":    kl_loss.item(),
        "total_loss": total_loss.item(),
        "mean_ratio": masked_mean(ratio, response_mask).item(),
    }
    return total_loss, pg_loss, kl_loss, stats


# 测试 GRPO 损失（on-policy 模式）
total_loss, pg_loss, kl_loss, stats = grpo_loss(
    policy_logprobs, ref_logprobs,
    token_advantages.detach(), response_mask,
    old_logprobs=None, kl_coef=0.04, use_clip=True
)

print()
print("== 最终统计 ==")
for k, v in stats.items():
    print(f"  {k}: {v:.6f}")

In [ ]:
# ============================================================
# 反向传播与参数更新示例
# 演示完整的 GRPO 单步训练流程
# ============================================================

optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-4)

print("== 单步 GRPO 训练 ==")
print()

# --- Step 1: 采样阶段（通常无梯度）---
print("[Step 1] 采样 G 个响应")
with torch.no_grad():
    sample_outputs = policy_model(full_ids)
    sample_logprobs = logprobs_from_logits(sample_outputs.logits, full_ids)  # 旧策略对数概率
    sample_rewards  = reward_model(full_ids, full_mask)                       # [G]

print(f"  采样奖励: {sample_rewards.tolist()}")

# --- Step 2: 计算组内相对优势 ---
print()
print("[Step 2] 计算组内相对优势")
adv = compute_group_advantages(sample_rewards)          # [G]
tok_adv = adv.unsqueeze(1).expand_as(full_ids).float() * response_mask  # [G, seq_len]

# --- Step 3: 前向计算（需要梯度）---
print()
print("[Step 3] 策略模型前向传播（梯度开启）")
new_outputs   = policy_model(full_ids)
new_logprobs  = logprobs_from_logits(new_outputs.logits, full_ids)

# --- Step 4: 计算 GRPO 损失 ---
print()
print("[Step 4] 计算 GRPO 损失")
loss, _, _, train_stats = grpo_loss(
    new_logprobs, ref_logprobs,
    tok_adv.detach(), response_mask,
    old_logprobs=sample_logprobs,  # off-policy 重要性采样
    kl_coef=0.04, use_clip=True
)

# --- Step 5: 反向传播 ---
print()
print("[Step 5] 反向传播")
optimizer.zero_grad()
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
print(f"  梯度范数: {grad_norm.item():.6f}")

# --- Step 6: 参数更新 ---
optimizer.step()
print()
print("[Step 6] 参数更新完成")
print()
print("== 训练统计 ==")
for k, v in train_stats.items():
    print(f"  {k}: {v:.6f}")

In [ ]:
# ============================================================
# GRPO vs PPO 核心对比总结
# ============================================================

summary = """
╔══════════════════════╦═══════════════════════════╦════════════════════════════╗
║        维度          ║           PPO             ║           GRPO             ║
╠══════════════════════╬═══════════════════════════╬════════════════════════════╣
║ 模型数量             ║ 4 (Actor/Critic/Ref/RM)   ║ 2 (Policy/Reference)       ║
║ 基线估计             ║ Critic 网络 V(s)          ║ 组内平均奖励 mean(r)       ║
║ 优势计算             ║ GAE (时间差分展开)        ║ (r_i - mean) / std         ║
║ 奖励位置             ║ 末尾token + KL惩罚        ║ 直接用组内相对奖励         ║
║ 显存开销             ║ 高（需加载Critic）        ║ 低（省去Critic）            ║
║ 适用场景             ║ 通用 RLHF                 ║ 可验证奖励（如数学/代码）  ║
║ 代表工作             ║ InstructGPT, Llama2-Chat  ║ DeepSeekMath, DeepSeek-R1  ║
╚══════════════════════╩═══════════════════════════╩════════════════════════════╝

GRPO 关键公式:
  优势: A_i = (r_i - mean_G(r)) / std_G(r)
  损失: L = -E[A_i * log π_θ(o_i|q)] + β * KL(π_θ || π_ref)
  
  （带裁剪变体）
  L = -E[min(A_i * ratio, A_i * clip(ratio, 1-ε, 1+ε))] + β * KL
"""

print(summary)